In [ ]:
"""
Baseline A — GradientBoosting + Typewell GR matching
=====================================================
Core idea:
  The typewell gives a GR template vs depth.
  We slide this template against the horizontal well's GR to estimate
  the current TVT. Features encode the match quality + local dip + position.

Expected ~15-30 ft RMSE on a cold start (no tuning).
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os, warnings, sys
from pathlib import Path
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error

warnings.filterwarnings('ignore')

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# ─────────────────────────────────────────────
# PATHS  (auto-detect Kaggle vs local)
# ─────────────────────────────────────────────
if os.path.exists('/kaggle/input'):
    BASE  = Path('/kaggle/input/competitions/rogii-wellbore-geology-prediction')
    TRAIN = BASE / 'train'
    TEST  = BASE / 'test'
    SUB   = BASE / 'sample_submission.csv'
    OUTPUT_DIR = Path('/kaggle/working')
else:
    BASE  = Path.cwd()
    TRAIN = BASE / 'train'
    TEST  = BASE / 'test'
    SUB   = BASE / 'sample_submission.csv'
    OUTPUT_DIR = BASE

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"BASE: {BASE}")
print(f"TRAIN exists: {TRAIN.exists()}")
print(f"TEST exists: {TEST.exists()}")
print(f"SUB exists: {SUB.exists()}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")

SEED      = 42
VAL_FRAC  = 0.2
GR_WIN    = 15   # rolling window for GR smoothing

# ─────────────────────────────────────────────
# LOADERS
# ─────────────────────────────────────────────
def load_wells(d):
    out = []

    for hf in sorted(Path(d).glob('*__horizontal_well.csv')):
        wid = hf.stem.replace('__horizontal_well','')

        tws = list(Path(d).glob(f'{wid}__typewell.csv'))
        if not tws:
            continue

        out.append((wid, pd.read_csv(hf), pd.read_csv(tws[0])))

    return out


def ps_idx(h):
    v = h['TVT_input'].notna()
    return int(v[v].index[-1]) if v.any() else 0


def tw_interp(tw, tvt_q):
    s = tw.sort_values('TVT').dropna(subset=['TVT','GR'])
    return np.interp(tvt_q, s['TVT'].values, s['GR'].values)

Loading training wells …
  Train: 619 wells | Val: 154 wells
Building training features …
  X_train shape: (3007768, 21)
Building validation features …
  X_val   shape: (776221, 21)

Training GradientBoostingRegressor baseline …



KeyboardInterrupt



In [ ]:
# ─────────────────────────────────────────────
# FEATURE ENGINEERING
# ─────────────────────────────────────────────
FEAT_COLS = None   # set on first call

def make_feats(h, tw):
    df = h.copy()
    ps = ps_idx(df)
    last_tvt = df.loc[ps, 'TVT_input']

    df['dMD'] = df['MD'].diff().fillna(0)
    for c in ['X','Y','Z']:
        if c in df.columns:
            df[f'd{c}'] = df[c].diff().fillna(0)

    gr = df['GR'].interpolate().bfill().ffill()
    df['gr']       = gr
    df['gr_mean']  = gr.rolling(GR_WIN, min_periods=1).mean()
    df['gr_std']   = gr.rolling(GR_WIN, min_periods=1).std().fillna(0)
    df['gr_d1']    = gr.diff().fillna(0)
    df['gr_d2']    = df['gr_d1'].diff().fillna(0)

    df['tvt_filled']    = df['TVT_input'].interpolate().bfill().ffill()
    df['tvt_last']      = last_tvt
    df['steps_from_ps'] = (np.arange(len(df)) - ps).clip(min=0)

    df['tw_gr']      = tw_interp(tw, df['tvt_filled'].values)
    df['tw_residual']= df['gr'] - df['tw_gr']
    df['tw_gr_last'] = tw_interp(tw, np.full(len(df), last_tvt))

    known = df.loc[:ps, ['MD','TVT_input']].dropna()
    dip = np.polyfit(known.tail(50)['MD'], known.tail(50)['TVT_input'], 1)[0] if len(known) > 2 else 0.0
    df['dip']        = dip
    df['tvt_extrap'] = last_tvt + dip * (df['MD'] - df.loc[ps, 'MD'])

    base = ['MD','dMD','gr','gr_mean','gr_std','gr_d1','gr_d2',
            'tvt_filled','tvt_last','steps_from_ps',
            'tw_gr','tw_residual','tw_gr_last','dip','tvt_extrap']
    xyz  = [c for c in ['X','Y','Z','dX','dY','dZ'] if c in df.columns]
    return df, base + xyz

In [ ]:
# ─────────────────────────────────────────────
# BUILD TRAIN / VAL ARRAYS
# ─────────────────────────────────────────────
def build_dataset(wells):
    Xs, ys, groups = [], [], []
    fcols = None
    for wid, h, tw in wells:
        ps = ps_idx(h)
        df, fc = make_feats(h, tw)
        fcols = fc
        zone  = df.iloc[ps+1:]
        if len(zone) == 0:
            continue
        Xs.append(zone[fc].values)
        ys.append(zone['TVT'].values)
        groups += [wid]*len(zone)
    X = np.vstack(Xs)
    y = np.concatenate(ys)
    med = np.nanmedian(X, axis=0)
    for j in range(X.shape[1]):
        X[np.isnan(X[:,j]), j] = med[j]
    return X, y, groups, fcols, med

print('Loading wells …')
all_w = load_wells(TRAIN)
np.random.seed(SEED); np.random.shuffle(all_w)
n_val = max(1, int(len(all_w)*VAL_FRAC))
val_w, tr_w = all_w[:n_val], all_w[n_val:]
print(f'  Train: {len(tr_w)}  Val: {len(val_w)}')

X_tr, y_tr, _, fcols, med = build_dataset(tr_w)
X_vl, y_vl, _, _,     _   = build_dataset(val_w)
for j in range(X_vl.shape[1]):
    X_vl[np.isnan(X_vl[:,j]), j] = med[j]

print(f'  X_train: {X_tr.shape}   X_val: {X_vl.shape}')

In [ ]:
# ─────────────────────────────────────────────
# MODEL — LightGBM (much faster than GradientBoosting)
# ─────────────────────────────────────────────
model = LGBMRegressor(
    n_estimators=600,
    learning_rate=0.04,
    max_depth=5,
    num_leaves=31,
    subsample=0.8,
    min_child_samples=10,
    random_state=SEED,
    verbose=-1,
    n_jobs=-1
)
print('\nTraining LightGBM …')
print(f'  Training on {len(X_tr)} samples with {X_tr.shape[1]} features')
sys.stdout.flush()
model.fit(X_tr, y_tr)
print('✓ Training complete!')

rmse = lambda a, b: np.sqrt(mean_squared_error(a, b))
print(f'  Train RMSE : {rmse(y_tr, model.predict(X_tr)):.4f} ft')
print(f'  Val   RMSE : {rmse(y_vl, model.predict(X_vl)):.4f} ft')

In [ ]:
# ─────────────────────────────────────────────
# FEATURE IMPORTANCE + VAL PLOT
# ─────────────────────────────────────────────
fi = pd.Series(model.feature_importances_, index=fcols).sort_values(ascending=False)
print('\nTop 15 features:')
print(fi.head(15).round(4).to_string())

fig, ax = plt.subplots(figsize=(8, 5))
fi.head(15).plot(kind='barh', ax=ax, color='#0077b6')
ax.invert_yaxis(); ax.set_title('Feature Importances — Baseline A (LightGBM)')
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(min(3,len(val_w)), 1, figsize=(14, 5*min(3,len(val_w))))
if min(3,len(val_w)) == 1:
    axes = [axes]
for ax, (wid, h, tw) in zip(axes, val_w[:3]):
    ps = ps_idx(h)
    df, fc = make_feats(h, tw)
    zone   = df.iloc[ps+1:]
    X_w    = zone[fc].values
    for j in range(X_w.shape[1]):
        X_w[np.isnan(X_w[:,j]),j] = med[j]
    pred   = model.predict(X_w)
    r      = rmse(zone['TVT'].values, pred)
    ax.plot(h.loc[:ps,'MD'],   h.loc[:ps,'TVT_input'], color='#e9c46a', lw=1.5, label='known')
    ax.plot(zone['MD'], zone['TVT'], color='#0077b6', lw=1.5, label='truth')
    ax.plot(zone['MD'], pred,         color='#e76f51', lw=1.5, ls='--', label='pred')
    ax.axvline(h.loc[ps,'MD'], color='red', ls=':', lw=1.2, label='PS')
    ax.set_title(f'{wid}  RMSE={r:.2f} ft')
    ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# ─────────────────────────────────────────────
# INFERENCE + SUBMISSION
# ─────────────────────────────────────────────
sample_sub = pd.read_csv(SUB)
sample_sub[['well_hash','md_int']] = sample_sub['id'].str.rsplit('_', n=1, expand=True)
sample_sub['md_int'] = sample_sub['md_int'].astype(int)
print('sample_sub sample:')
print(sample_sub.head())
print('sample_sub unique well_hash count:', sample_sub['well_hash'].nunique())
print('sample_sub count per well:')
print(sample_sub['well_hash'].value_counts())

# load test wells and show debug info
test_wells = load_wells(TEST)
print('Loaded test wells:', len(test_wells))
print('First test well ids:', [wid for wid, _, _ in test_wells[:10]])

sample_sub['tvt'] = 0.0
well_counts = {}
missing_wells = []
for wid, h, tw in test_wells:
    ps = ps_idx(h)
    df, fc = make_feats(h, tw)
    zone = df.iloc[ps+1:].copy()
    if zone.empty:
        print('Skipping empty test zone for', wid)
        continue
    X_t = zone[fc].values
    for j in range(X_t.shape[1]):
        X_t[np.isnan(X_t[:,j]), j] = med[j]
    preds = model.predict(X_t)
    wh = wid[:8] if len(wid) >= 8 else wid

    sub_idx = sample_sub[sample_sub['well_hash'] == wh].sort_values('md_int').index
    print(f'Well {wh}: sample rows={len(sub_idx)} preds={len(preds)} ' \
          f'MD range=({zone["MD"].iat[0]}, {zone["MD"].iat[-1]}) ' \
          f'first sample md_int=({sample_sub.loc[sub_idx, "md_int"].iat[0]}, ' \
          f'{sample_sub.loc[sub_idx, "md_int"].iat[-1]})')
    if len(sub_idx) == len(preds):
        sample_sub.loc[sub_idx, 'tvt'] = preds
        well_counts[wh] = len(preds)
    elif len(sub_idx) > 0:
        print(f'Count mismatch for well {wh}: sample_sub={len(sub_idx)} preds={len(preds)}')
        missing_wells.append(wh)
        min_len = min(len(sub_idx), len(preds))
        sample_sub.loc[sub_idx[:min_len], 'tvt'] = preds[:min_len]

print('Assigned predictions for wells:', well_counts)
if missing_wells:
    print('Fallback assignment used for wells:', missing_wells)

missing = int((sample_sub['tvt'] == 0.0).sum())
print('Missing predictions after assignment:', missing, 'out of', len(sample_sub))

out = sample_sub[['id','tvt']]
submission_file = OUTPUT_DIR / 'submission.csv'
out.to_csv(submission_file, index=False)
print(f'\nSubmission written to: {submission_file}')
print(f'File exists: {submission_file.exists()}')
print(f'{len(out)} rows')
print(out.head())
print(out['tvt'].describe())